# Cross-model self-recognition — Llama vs Qwen

**Question.** When a model judges *"which text did I write?"*, is it recognizing its
**own model** (Llama vs Qwen) or its **own persona** (the system prompt it was under)?
We separate the two with four cross-model **test cases** (the `foil_type`):

| # | foil_type | evaluator is shown | isolates | chance |
|--|---|---|---|--|
| 1 | `diff_model_same_persona` | **self** vs *(other model, same persona)* | **model** recognition | 0.5 |
| 2 | `same_model_diff_persona` | **self** vs *(same model, other persona)* | **persona** recognition | 0.5 |
| 3 | `diff_model_diff_persona` | **self** vs *(other model, other persona)* | both cues | 0.5 |
| 4 | `persona_vs_model` | *(other model, same persona)* vs *(same model, other persona)* — **neither is self** | persona-anchor **vs** model-anchor | 0.5 |

Cases 1–3 are *self-recognition* → accuracy = P(pick own text). Case 4 is an
*attribution probe* (neither text is its own) → accuracy = **persona-anchor rate**
(>0.5 claims the same-persona text ⇒ anchors on persona; <0.5 ⇒ anchors on model).
Conditions: **active** (persona induced) vs **neutral** (baseline). Counterbalanced,
so chance = 0.5. **All bar plots carry 95% CIs.**

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import sys, pathlib
_root = pathlib.Path.cwd().resolve()
while not (_root / "core").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import logging; logging.basicConfig(level=logging.INFO)
import pandas as pd, matplotlib.pyplot as plt
import experiments.self_recognition.analyze_behavior_helpers as B      # stats + plotters (95% CI)
import experiments.self_recognition.analyze_cross_model_helpers as X   # cross-model load/agg/plots
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)

EVAL_DIR = "cross_model_v1"
MODELS = ["meta-llama/Llama-3.1-8B-Instruct", "Qwen/Qwen2.5-32B-Instruct"]
df = X.load_cross_model(EVAL_DIR, MODELS)

# Ladder ordering + pretty labels for the "test cases" (numeric prefix => sorts as a ladder).
FOIL_ORDER = ["diff_model_same_persona","same_model_diff_persona","diff_model_diff_persona","persona_vs_model"]
# 1-3 are SELF-vs-FOIL (parenthetical = the FOIL; self is the other text).
# 4 is the anchor probe: NEITHER text is self, so both sources are named.
FOIL_TITLE = {
 "diff_model_same_persona":"1 foil:\nother model,\nsame persona",
 "same_model_diff_persona":"2 foil:\nsame model,\nother persona",
 "diff_model_diff_persona":"3 foil:\nother model,\nother persona",
 "persona_vs_model":"4 anchor probe:\n(other model, same-P)\nvs (same model, other-Q)"}
df["case_label"] = df.foil_type.map(FOIL_TITLE).fillna(df.foil_type)

print(f"{len(df):,} trials | evaluators: {sorted(df.evaluator_slug.unique())}")
print("foil_types present:", df.foil_type.value_counts().to_dict())
# Warn if a probe/foil is missing for an evaluator (e.g. persona_vs_model not run for a model).
cov = df.groupby("evaluator_slug").foil_type.apply(lambda s: sorted(s.unique()))
for ev, fts in cov.items():
    missing = [f for f in FOIL_ORDER if f not in fts]
    if missing: print(f"  ⚠ {ev} is MISSING: {missing}")

163,728 trials | evaluators: ['Llama-3-1-8B-Instruct', 'Qwen2-5-32B-Instruct']
foil_types present: {'same_model_diff_persona': 51840, 'diff_model_diff_persona': 51840, 'persona_vs_model': 51840, 'diff_model_same_persona': 8208}


## 0 · Balance check

Counterbalancing makes correct=A ≈ 0.5 in every cell by construction. Deviation ⇒
an enumeration bug. (Deterministic invariant, so a table, not a CI plot.)

In [ ]:
bal = (df.assign(cA=df.correct_answer.eq("A"))
         .groupby(["evaluator_slug","foil_type","condition"])
         .agg(frac_correct_A=("cA","mean"), n=("trial_id","size")))
display(bal.round(3))

## 3 · HEADLINE — the cross-model ladder (all test cases)

All four test cases on one axis, **active** condition, per evaluator. Read it as a
ladder of cues:

- **1 model cue** clears 0.5 ⇒ the model recognizes its own *model* fingerprint.
- **2 persona cue** clears 0.5 ⇒ it recognizes its own *persona*.
- **3 both cues** is the easy ceiling.
- **4 anchor probe** (neither text is self): >0.5 ⇒ **persona-anchored**, <0.5 ⇒ **model-anchored**.

Chance = 0.5 everywhere. *(Grouped bars = the two evaluators; 95% CI error bars.)*

In [ ]:
act = df[df.condition=="active"]
lad = X.agg(act, by=["case_label","evaluator_slug"]).rename(columns={"case_label":"label"})
display(lad[["label","evaluator_slug","n","accuracy","ci_lo","ci_hi","p_vs_chance","mean_prob_correct"]]
        .sort_values(["label","evaluator_slug"]).round(3))
B.plot_contrast_by_category(lad, group="evaluator_slug",
    title="Cross-model ladder (active) — argmax accuracy  (chance=0.5)"); plt.show()
B.plot_contrast_by_category(lad, group="evaluator_slug", value="mean_prob_correct",
    lo_col="prob_ci_lo", hi_col="prob_ci_hi", ylabel="mean P(correct)",
    title="Cross-model ladder (active) — graded logprob mass"); plt.show()

### 3b · The ladder by persona category — where the cue lives

The verdict cut (mirrors the single-model §3b). For each evaluator, the ladder split
by the evaluator persona's category. **Model recognition** that only appears in
obvious-style categories (calibration/confound) is style leaking through; a signal in
**suppression / near_twin** is the interesting one. *(95% CIs.)*

In [ ]:
for ev in sorted(act.evaluator_slug.unique()):
    sub = act[act.evaluator_slug==ev]
    cat = X.agg(sub, by=["case_label","evaluator_coarse"]).rename(columns={"case_label":"label"})
    if cat.empty: continue
    B.plot_contrast_by_category(cat, title=f"{ev} — ladder by persona category (active)"); plt.show()
    display(cat.pivot_table(index="evaluator_coarse", columns="label", values="accuracy", observed=True).round(3))

### 3c · Active vs neutral — the active-state effect

The cross-model analogue of case7 − case10: how much does *inducing the persona* add
over the neutral baseline, per test case? Diverging bars around 0 with a **two-sample
95% CI**; a CI clearing 0 = a real active-state gain. For self-recognition cases a
positive gain concentrated in suppression/near_twin would be the introspection result.

In [ ]:
sur=[]
for ft in FOIL_ORDER:
    if ft not in df.foil_type.unique(): continue
    s = X.active_vs_neutral(df, foil_type=ft, by=("evaluator_slug",)); s["foil_type"]=ft
    sur.append(s)
sur = pd.concat(sur, ignore_index=True)
sur["label"] = sur.evaluator_slug.str.replace("-Instruct","",regex=False)+"\n"+sur.foil_type.map(FOIL_TITLE).str.split("\n").str[0]
display(sur[["evaluator_slug","foil_type","acc_neutral","acc_active","acc_surplus","acc_surplus_lo","acc_surplus_hi"]].round(3))
X.plot_surplus(sur, title="Active − neutral surplus by test case (argmax) — 95% CI"); plt.show()
X.plot_surplus(sur, value="prob_surplus", lo_col="prob_surplus_lo", hi_col="prob_surplus_hi",
               ylabel="active − neutral ΔP", color="#55A868",
               title="Active − neutral surplus by test case (graded) — 95% CI"); plt.show()

# Persona-recognition active gain by category (the privileged-access lens).
sc = X.active_vs_neutral(df, foil_type="same_model_diff_persona", by=("evaluator_slug","evaluator_coarse"))
if not sc.empty:
    for ev in sorted(sc.evaluator_slug.unique()):
        t = sc[sc.evaluator_slug==ev].copy(); t["label"]=t.evaluator_coarse
        X.plot_surplus(t, title=f"{ev}: persona-rec active−neutral by category — 95% CI"); plt.show()

## 4 · Model recognition deep-dive (`diff_model_same_persona`)

Only the model differs (same persona both sides), so this isolates the model
fingerprint. Active vs neutral: if neutral is already high, the fingerprint is in the
text and doesn't need the persona active. *(95% CIs.)*

In [ ]:
mr = X.model_recognition(df, by=("evaluator_slug","condition"))
mr["label"] = mr.evaluator_slug + "\n" + mr.condition
display(mr[["evaluator_slug","condition","n","accuracy","ci_lo","ci_hi","p_vs_chance","mean_prob_correct"]].round(3))
B.plot_accuracy(mr, title="Model recognition — argmax accuracy"); plt.show()
B.plot_accuracy(mr, value="mean_prob_correct", lo_col="prob_ci_lo", hi_col="prob_ci_hi",
                ylabel="mean P(correct)", color="#55A868", title="Model recognition — graded"); plt.show()

NEUTRAL_PERSONA = "default_neutral"
npb = df[(df.foil_type=="diff_model_same_persona") & (df.evaluator_persona==NEUTRAL_PERSONA)]
if not npb.empty:
    nt = X.agg(npb, by=["evaluator_slug","condition"]); nt["label"]=nt.evaluator_slug+"\n"+nt.condition
    display(nt[["evaluator_slug","condition","n","accuracy","ci_lo","ci_hi","mean_prob_correct"]].round(3))
    B.plot_accuracy(nt, title=f"Neutral-persona model recognition ({NEUTRAL_PERSONA})"); plt.show()

# Per-persona heatmap (active).
grid = X.agg(df[(df.foil_type=="diff_model_same_persona") & (df.condition=="active")],
             by=["evaluator_slug","evaluator_persona"])
if not grid.empty:
    import numpy as np
    piv = grid.pivot_table(index="evaluator_persona", columns="evaluator_slug", values="accuracy", observed=True)
    fig, ax = plt.subplots(figsize=(5, max(4,0.32*len(piv))))
    im = ax.imshow(piv.values, cmap="RdBu_r", vmin=0.2, vmax=0.8, aspect="auto")
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns, fontsize=8)
    ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index, fontsize=7)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v=piv.values[i,j]
            if not np.isnan(v): ax.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=6)
    fig.colorbar(im, ax=ax, label="accuracy (0.5=chance)"); ax.set_title("Model recognition by persona (active)", fontsize=10)
    plt.tight_layout(); plt.show()

## 5 · Persona-vs-model anchor probe (`persona_vs_model`)

*(Only if you ran `self_recognition_cross_model_persona_vs_model`.)* Neither text is
the model's own: same-persona/other-model vs different-persona/same-model. Accuracy =
**persona-anchor rate**: **>0.5** → claims the same-persona text (**anchors on persona**),
**<0.5** → claims the same-model text (**anchors on model**), 0.5 = no lean. *(95% CIs;
dashed line 0.5.)*

In [ ]:
pvm = df[df.foil_type=="persona_vs_model"]
if pvm.empty:
    print("No persona_vs_model rows — run: python run.py self_recognition_cross_model_persona_vs_model")
else:
    pa = X.persona_anchor(df, condition="active"); pn = X.persona_anchor(df, condition="neutral")
    pa["label"]=pa.evaluator_slug+"\nactive"; pn["label"]=pn.evaluator_slug+"\nneutral"
    both = pd.concat([pa,pn], ignore_index=True)
    display(both[["evaluator_slug","label","n","accuracy","ci_lo","ci_hi","p_vs_chance","mean_prob_correct"]].round(3))
    B.plot_accuracy(both, title="Persona-anchor rate (>0.5 persona, <0.5 model)"); plt.show()
    pac = X.agg(pvm[pvm.condition=="active"], by=["evaluator_slug","evaluator_coarse"]).rename(columns={"evaluator_slug":"label"})
    if not pac.empty:
        B.plot_contrast_by_category(pac, title="Persona-anchor rate by category (active)"); plt.show()

## 6 · Position-bias diagnostic

Accuracy split by `text_order`. A large gap = the model attributes the **first** text
to itself regardless of content. Counterbalancing cancels this in the mean (equal n
per order), but the gap explains why a mean can sit near/below 0.5. *(95% CIs.)*

In [ ]:
import numpy as np
a = df[df.condition=="active"].copy()
pos = X.agg(a, by=["evaluator_slug","foil_type","text_order"])
pos["label"] = a.foil_type.map(FOIL_TITLE); pos["label"] = pos.foil_type.map(lambda f: FOIL_TITLE[f].split("\n")[0])
pos["label"] = pos.evaluator_slug.str.replace("-Instruct","",regex=False)+"\n"+pos.label
B.plot_contrast_by_category(pos, group="text_order",
    title="Accuracy by text position (active) — gap = position bias (95% CI)"); plt.show()
# net lean, bug-free (uses is_correct + text-slot identity)
a["target_text1"] = a.answer_mapping.map(lambda m: m if isinstance(m,dict) else __import__("json").loads(m)).map(lambda m: m["A"]=="text1")
# picks text1 = correct when target in text1 else incorrect
a["picks_text1"] = np.where(a.target_text1, a.is_correct, ~a.is_correct.astype(bool))
print("P(attributes Text 1 to itself)  [0.5 = unbiased]:")
print(a.groupby("evaluator_slug").picks_text1.mean().round(3).to_string())

## 7 · Introspection vs content inference — cases 6/7/8

**One data source:** everything is in `cross_model_v1`. It already held **case 7**
(active, no description = INTROSPECTION) across all foils + the anchor probe; running
`self_recognition_cross_model_cases` appends **case 6** (neutral, target described =
CONTENT INFERENCE) and **case 8** (neutral, both described = CEILING). Rows from the
original run have no `base_case` and are treated as **case7** on load.

| case | condition | shows | measures |
|--|---|---|---|
| **7** | active | no description | recognize own text *from being the persona* → **introspection** |
| **6** | neutral | target persona described | match text→description *without being it* → **content inference** |
| **8** | neutral | both described (attribute-both) | **content-inference ceiling** |

**Headline: `case7(active) − case6(neutral)` = introspective surplus over content
inference** — the signal lives in **suppression / near_twin**. Cases 6 & 8 describe a
persona to disambiguate, so they run only where the persona differs
(`same_model_diff_persona`, `diff_model_diff_persona`). ⚠ Case 8 is attribute-both
(harder) — read it as the ceiling; compare 7 vs 6 directly. *(95% CIs throughout.)*

In [ ]:
# All from the single df loaded in cell 1 (cross_model_v1). Guarded until case6/8 exist.
CASE_FOILS = ["same_model_diff_persona", "diff_model_diff_persona"]  # where 7/6/8 all exist
HAVE_CASES = df.base_case.isin(["case6", "case8"]).any()
if not HAVE_CASES:
    print("No case6/case8 rows in cross_model_v1 yet — run: python run.py self_recognition_cross_model_cases")
else:
    cases = df[df.base_case.isin(["case6","case7","case8"])]
    print("base_cases present:", df.base_case.value_counts().to_dict())
    print("coverage (rows per base_case × foil_type; case7 shown in its native active condition):")
    display(cases.groupby(["base_case","foil_type"]).size().unstack(fill_value=0))
    balc = (cases.assign(cA=cases.correct_answer.eq("A"))
                 .groupby(["evaluator_slug","base_case","foil_type"]).cA.mean().rename("frac_correct_A"))
    display(balc.round(3).to_frame())

### 7a · The 7/6/8 ladder per foil type

Accuracy for the three questions side by side, per evaluator (each case in its native
condition). Chance = 0.5. Read **7 vs 6** (introspection vs content inference); **8** is
the ceiling. Argmax + graded, 95% CIs.

In [ ]:
if HAVE_CASES:
    for ft in CASE_FOILS:
        lad = X.case_ladder(df, foil_type=ft, by=("evaluator_slug",))
        if lad.empty:
            print(f"{ft}: no data"); continue
        display(lad[["base_case","evaluator_slug","n","accuracy","ci_lo","ci_hi","p_vs_chance","mean_prob_correct"]].round(3))
        ftl = X.FOIL_LABELS[ft].replace(chr(10)," ")
        B.plot_contrast_by_category(lad, group="evaluator_slug",
            title=f"Cases 7/6/8 — {ftl} — argmax accuracy (95% CI)"); plt.show()
        B.plot_contrast_by_category(lad, group="evaluator_slug", value="mean_prob_correct",
            lo_col="prob_ci_lo", hi_col="prob_ci_hi", ylabel="mean P(correct)",
            title=f"Cases 7/6/8 — {ftl} — graded logprob mass (95% CI)"); plt.show()

### 7b · The ladder by persona category — the verdict

For each evaluator × foil type, the 7/6/8 ladder split by persona category. If **case 7
(introspection)** clears **case 6 (content inference)** specifically in **suppression /
near_twin**, that's the introspection signal; if 7≈6 and both are high only in
calibration/confound, it's style. *(95% CIs.)*

In [ ]:
if HAVE_CASES:
    for ft in CASE_FOILS:
        for ev in sorted(df.evaluator_slug.unique()):
            cat = X.case_ladder(df[df.evaluator_slug==ev], foil_type=ft, by=("evaluator_coarse",))
            if cat.empty: continue
            ftl = X.FOIL_LABELS[ft].replace(chr(10)," ")
            B.plot_contrast_by_category(cat, group="evaluator_coarse",
                title=f"{ev} — cases 7/6/8 by category — {ftl}"); plt.show()

### 7c · Introspective surplus (case 7 − case 6)

Active-introspection accuracy minus content-inference accuracy, per evaluator and by
persona category, with a **two-proportion 95% CI**. Bars clearing 0 in **suppression /
near_twin** = introspection beyond what the description alone supports.

In [ ]:
if HAVE_CASES:
    for ft in CASE_FOILS:
        s = X.introspection_surplus(df, foil_type=ft, by=("evaluator_slug",))
        if s.empty:
            print(f"{ft}: need both case7 and case6"); continue
        s["label"] = s.evaluator_slug.str.replace("-Instruct","",regex=False)
        ftl = X.FOIL_LABELS[ft].replace(chr(10)," ")
        print(f"=== {ftl} ==="); display(s.round(3))
        X.plot_surplus(s, title=f"Introspective surplus  case7 − case6  — {ftl} (95% CI)"); plt.show()
        sc = X.introspection_surplus(df, foil_type=ft, by=("evaluator_slug","evaluator_coarse"))
        for ev in sorted(sc.evaluator_slug.unique()):
            t = sc[sc.evaluator_slug==ev].copy(); t["label"] = t.evaluator_coarse
            if not t.empty:
                X.plot_surplus(t, title=f"{ev}: case7 − case6 by category — {ftl} (95% CI)"); plt.show()

### 7d · Master table — base_case × foil_type × evaluator

Every cell's accuracy in one grid (chance = 0.5; each case in its native condition).
Includes case 7 on `diff_model_same_persona` (pure model recognition) for reference.

In [ ]:
if HAVE_CASES:
    cases = df[df.base_case.isin(["case6","case7","case8"])].copy()
    # keep each case in its native condition so case7's neutral floor doesn't double-count
    cases = cases[cases.apply(lambda r: r.condition == X.NATIVE_CONDITION.get(r.base_case, r.condition), axis=1)]
    m = X.agg(cases, by=["foil_type","base_case","evaluator_slug"])
    print("Argmax accuracy:")
    display(m.pivot_table(index=["foil_type","base_case"], columns="evaluator_slug",
                          values="accuracy", observed=True).round(3))
    print("Graded mean P(correct):")
    display(m.pivot_table(index=["foil_type","base_case"], columns="evaluator_slug",
                          values="mean_prob_correct", observed=True).round(3))